# AR Door Visualizer - 3D Generation Test

Test TripoSR image-to-3D reconstruction with door photos.

**Instance:** ml.g4dn.xlarge (NVIDIA T4, 16GB VRAM)

**⚠️ STOP THIS NOTEBOOK INSTANCE WHEN DONE (~$0.94/hr)**

## 1. Install Dependencies

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 -q
!pip install transformers trimesh einops omegaconf pillow rembg -q

## 2. Clone TripoSR

In [ ]:
!git clone https://github.com/VAST-AI-Research/TripoSR.git
%cd TripoSR
!pip install -r requirements.txt -q

## 3. Upload Your Door Photo

Upload your door photo to this notebook (use the upload button in Jupyter), or download from S3:

In [ ]:
# Option A: If you uploaded the file to the notebook
INPUT_IMAGE = "door.jpg"  # Change to your filename

# Option B: Download from S3 (if you uploaded samples there)
# !aws s3 cp s3://your-bucket/samples/door.jpg door.jpg
# INPUT_IMAGE = "door.jpg"

from PIL import Image
img = Image.open(INPUT_IMAGE)
img.thumbnail((512, 512))
display(img)

## 4. (Optional) Remove Background

TripoSR works much better with a clean white background.

In [ ]:
from rembg import remove
from PIL import Image
import io

input_img = Image.open(INPUT_IMAGE)
output_img = remove(input_img)

# Add white background
white_bg = Image.new('RGBA', output_img.size, (255, 255, 255, 255))
white_bg.paste(output_img, mask=output_img.split()[3])
white_bg = white_bg.convert('RGB')
white_bg.save('door_clean.png')

display(white_bg)
INPUT_IMAGE = 'door_clean.png'
print('Background removed successfully!')

## 5. Run TripoSR (Image → 3D)

In [ ]:
!python run.py {INPUT_IMAGE} \
    --output-dir output/ \
    --model-save-format glb \
    --render

## 6. Check Output

In [ ]:
import os

output_dir = 'output/0/'
files = os.listdir(output_dir)
print(f'Output files: {files}')

# Check GLB file size
glb_path = os.path.join(output_dir, 'mesh.glb')
if os.path.exists(glb_path):
    size_mb = os.path.getsize(glb_path) / (1024*1024)
    print(f'GLB file size: {size_mb:.1f} MB')
    print(f'\n✅ Success! Download {glb_path} and view at:')
    print('   https://gltf-viewer.donmccurdy.com/')
else:
    print('❌ GLB not found. Check errors above.')

## 7. View Rendered Preview (if --render was used)

In [ ]:
from IPython.display import display, Image as IPImage
import glob

renders = glob.glob('output/0/*.png')
for r in renders:
    print(r)
    display(IPImage(filename=r, width=400))

## 8. Upload GLB to S3 (for use in the app)

In [ ]:
import boto3

BUCKET = 'ar-door-visualizer-assets'  # Change to your bucket
KEY = 'models/test-door.glb'

# Uncomment to upload:
# s3 = boto3.client('s3', region_name='ap-southeast-1')
# s3.upload_file(glb_path, BUCKET, KEY, ExtraArgs={'ContentType': 'model/gltf-binary'})
# print(f'Uploaded to s3://{BUCKET}/{KEY}')

## ⚠️ DONE? STOP THE NOTEBOOK INSTANCE!

Go to SageMaker Console → Notebook Instances → Stop `ar-door-3d-test`

Or run: `aws sagemaker stop-notebook-instance --notebook-instance-name ar-door-3d-test --region ap-southeast-1`